# Issue #31: check `predict.py` on Colab

**Runtime > Run all.** The only manual step is the Google Drive sign-in pop-up.

This runs every `predict_check.py` check in one go:
1. **GPU, bf16** (the acceptance test, B1): batch sizes 8 and 1, max |diff| <= 1e-3 against the saved validation probabilities.
2. **CPU, fp32** (B2) on the same machine (`--device cpu`, so no runtime switch): max |diff| <= 2e-2 plus yes/no agreement at the cutoffs, with CPU timings.

Both runs also do check A (chunk boundaries, raw offsets) and the smoke test C on an unseen contract.

Needs: an **A100 or L4** GPU (bf16; a T4 falls back to fp32 and can't run B1), and the `cuad_finetune` folder in your Drive
(if it was shared with you, add a shortcut to it in My Drive). Optional: upload a non-CUAD contract to `/content/my_contract.txt`
for the smoke test; otherwise it uses the text of `test.json` contract 0 (text only, no labels, no metrics).

In [1]:
# settings
BRANCH = "albert-issue31"
REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1N-contract-review-challenge.git"
DRIVE_DIR = "/content/drive/MyDrive/cuad_finetune"
MODEL_ROOT = f"{DRIVE_DIR}/runs"                         # holds <run>/model/ for both ensemble runs
CHUNKS_DF = f"{DRIVE_DIR}/data/chunks_df.parquet"        # training chunks, for the exact chunk text/offset check
SMOKE_TEXT = "/content/my_contract.txt"                  # optional unseen contract

In [2]:
import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > A100 or L4"
name, (major, minor) = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
print(f"GPU: {name} (compute capability {major}.{minor})")
if major < 8:
    print("WARNING: this GPU has no fast bf16, so the GPU run will use fp32 and the 1e-3 acceptance test (B1) won't "
          "apply. Switch to an A100 or L4 for the real check.")

GPU: NVIDIA L4 (compute capability 8.9)


In [3]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
for path in [Path(MODEL_ROOT) / "modernbert_v2_biasinit" / "model" / "config.json",
             Path(MODEL_ROOT) / "modernbert_v1_6ep" / "model" / "config.json",
             Path(CHUNKS_DF)]:
    print(("found    " if path.exists() else "MISSING  ") + str(path))

Mounted at /content/drive
found    /content/drive/MyDrive/cuad_finetune/runs/modernbert_v2_biasinit/model/config.json
found    /content/drive/MyDrive/cuad_finetune/runs/modernbert_v1_6ep/model/config.json
found    /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet


In [4]:
import os
REPO_DIR = "/content/repo"
if not os.path.exists(REPO_DIR):
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --quiet origin {BRANCH} && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} reset --quiet --hard origin/{BRANCH}
!git -C {REPO_DIR} log --oneline -1
!pip install -q git+https://github.com/brandonstarxel/chunking_evaluation.git backoff
%cd {REPO_DIR}/notebooks

7919a7c (HEAD -> albert-issue31, origin/albert-issue31) Add Colab notebook for running predict_check.py with GPU and CPU tests
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dat

In [5]:
import subprocess, sys, time

env = {**os.environ, "CUAD_MODEL_ROOT": MODEL_ROOT}
smoke = ["--smoke-text", SMOKE_TEXT] if os.path.exists(SMOKE_TEXT) else ["--smoke-test-json", "0"]
chunks = ["--chunks-df", CHUNKS_DF] if os.path.exists(CHUNKS_DF) else []
LOG_DIR = Path("/content/predict_check_logs")
LOG_DIR.mkdir(exist_ok=True)


def run_check(label, extra_args):
    # streams predict_check.py's output into the cell and keeps a copy in LOG_DIR
    cmd = [sys.executable, "predict_check.py", *chunks, *smoke, *extra_args]
    print("$ " + " ".join(cmd), flush=True)
    start = time.time()
    log = LOG_DIR / f"{label}.txt"
    with open(log, "w") as f:
        with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
            for line in proc.stdout:
                print(line, end="")
                f.write(line)
    print()
    print(f"[{label}] exit code {proc.returncode} after {(time.time() - start) / 60:.1f} min, log: {log}")
    return proc.returncode


results = {}

### 1. GPU, bf16 autocast (B1, acceptance), batch sizes 8 and 1

In [6]:
results["GPU bf16 (B1)"] = run_check("gpu_bf16", ["--batch-sizes", "8", "1"])

$ /usr/bin/python3 predict_check.py --chunks-df /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet --smoke-test-json 0 --batch-sizes 8 1
torch 2.14.1+cu130, transformers 5.22.1 (runs trained with 2.11.0+cu130 / 5.20.1); device cuda (NVIDIA L4)
validation contracts: ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT (1 chunks); RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT (15 chunks); AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1 (104 chunks)

=== A. chunk boundaries ===
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: chunk ids match val_probs (1)
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: text/start/end match training's chunks_df
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: raw offsets reproduce every chunk (1)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEME: chunk ids match val_probs (15)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEME: text/start

### 2. CPU, fp32 (B2), on this machine's CPU. Slower: a few minutes for the 3 contracts with 2 models

In [7]:
results["CPU fp32 (B2)"] = run_check("cpu_fp32", ["--device", "cpu", "--precision", "fp32"])

$ /usr/bin/python3 predict_check.py --chunks-df /content/drive/MyDrive/cuad_finetune/data/chunks_df.parquet --smoke-test-json 0 --device cpu --precision fp32
torch 2.14.1+cu130, transformers 5.22.1 (runs trained with 2.11.0+cu130 / 5.20.1); device cpu
validation contracts: ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT (1 chunks); RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT (15 chunks); AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1 (104 chunks)

=== A. chunk boundaries ===
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: chunk ids match val_probs (1)
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: text/start/end match training's chunks_df
  [PASS] ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT: raw offsets reproduce every chunk (1)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEME: chunk ids match val_probs (15)
  [PASS] RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEME: text/start/

### Summary

In [8]:
for label, code in results.items():
    print(f"{label:16s} {'PASS' if code == 0 else 'FAIL (see its output above)'}")
print()
print(f"logs in {LOG_DIR}: " + ", ".join(p.name for p in sorted(LOG_DIR.glob('*.txt'))))
assert all(code == 0 for code in results.values()), "at least one check failed"

GPU bf16 (B1)    FAIL (see its output above)
CPU fp32 (B2)    FAIL (see its output above)

logs in /content/predict_check_logs: cpu_fp32.txt, gpu_bf16.txt


AssertionError: at least one check failed